<h1 style="margin:0;color:#3776AB;">🐍 Pythonize com Tomane | #024</h1>

### Web scraping: comparar preços de livros com Python

#### 💡 Sabias que podes visitar várias páginas de um catálogo, comparar preços e guardar os resultados automaticamente?

No code de hoje, vou buscar livros do [Books to Scrape](https://books.toscrape.com/), um site de demonstração para praticar web scraping. Vamos recolher título, preço, avaliação e ligação, encontrar os livros mais baratos e criar dois gráficos. 📚


#### 1. Preparar as bibliotecas

Executa a instalação se ainda não tens estas bibliotecas no teu ambiente. Depois, importa o que vamos usar.


In [ ]:
%pip install requests beautifulsoup4 matplotlib


In [ ]:
import csv
import time
from collections import Counter
from pathlib import Path
from urllib.parse import urljoin

import matplotlib.pyplot as plt
import requests
from bs4 import BeautifulSoup

print("Tudo pronto para começar! 🐍")


#### 2. Escolher o catálogo e o limite

Neste exemplo, vou visitar duas páginas. Se quiseres explorar mais livros, basta aumentar `MAX_PAGINAS`.


In [ ]:
URL_INICIAL = "https://books.toscrape.com/"
MAX_PAGINAS = 2
PAUSA_SEGUNDOS = 1
LIMITE_PRECO = 20.00  # libras (£)

print(f"Vou consultar até {MAX_PAGINAS} páginas do catálogo.")


#### 3. Recolher título, preço, avaliação e ligação

Cada livro aparece num bloco HTML. Vamos percorrer esses blocos e usar o botão «next» para avançar. O preço é convertido para número, para podermos comparar os livros a seguir.


In [ ]:
livros = []
proxima_url = URL_INICIAL
sessao = requests.Session()
sessao.headers.update({"User-Agent": "PythonizeComTomane/1.0 (tutorial educativo)"})
estrelas = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

for numero_pagina in range(1, MAX_PAGINAS + 1):
    if proxima_url is None:
        break

    resposta = sessao.get(proxima_url, timeout=15)
    resposta.raise_for_status()
    pagina = BeautifulSoup(resposta.text, "html.parser")
    blocos = pagina.select("article.product_pod")

    if not blocos:
        raise ValueError("Não encontrei livros. Confere o endereço e os seletores da página.")

    for bloco in blocos:
        ligacao = bloco.select_one("h3 a")
        texto_preco = bloco.select_one("p.price_color").get_text(strip=True)
        avaliacao = bloco.select_one("p.star-rating")
        nota = next((estrelas[nome] for nome in avaliacao.get("class", []) if nome in estrelas), None)

        livros.append({
            "titulo": ligacao["title"],
            "preco_gbp": float(texto_preco.replace("£", "").strip()),
            "avaliacao": nota,
            "url": urljoin(resposta.url, ligacao["href"]),
        })

    print(f"Página {numero_pagina}: {len(blocos)} {'livro encontrado' if len(blocos) == 1 else 'livros encontrados'}")
    seguinte = pagina.select_one("li.next a")
    proxima_url = urljoin(resposta.url, seguinte["href"]) if seguinte else None

    if proxima_url and numero_pagina < MAX_PAGINAS:
        time.sleep(PAUSA_SEGUNDOS)

print(f"Total: {len(livros)} {'livro' if len(livros) == 1 else 'livros'}")


#### 4. Encontrar os livros abaixo de £20

Vou filtrar os livros dentro do limite definido e ordenar do mais barato para o mais caro. Podes mudar `LIMITE_PRECO` no passo 2.


In [ ]:
mais_baratos = sorted(
    (livro for livro in livros if livro["preco_gbp"] <= LIMITE_PRECO),
    key=lambda livro: livro["preco_gbp"]
)

print(f"Encontrei {len(mais_baratos)} {'livro' if len(mais_baratos) == 1 else 'livros'} até £{LIMITE_PRECO:.2f}.")
for livro in mais_baratos[:5]:
    print(f"£{livro['preco_gbp']:.2f} | {livro['titulo']} | {livro['avaliacao']} estrelas")


#### 5. Guardar o catálogo em CSV

O CSV fica na mesma pasta do notebook. Cada linha traz o título, o preço em libras, a avaliação e a ligação do livro.


In [ ]:
ficheiro = Path("Pythonize_024_Livros.csv")

with ficheiro.open("w", encoding="utf-8-sig", newline="") as saida:
    escritor = csv.DictWriter(saida, fieldnames=["titulo", "preco_gbp", "avaliacao", "url"])
    escritor.writeheader()
    escritor.writerows(livros)

print(f"CSV guardado em: {ficheiro.resolve()}")


#### 6. Visualizar os preços e as avaliações 📊

O primeiro gráfico mostra como os preços se distribuem. O segundo conta quantos livros receberam cada avaliação.


In [ ]:
precos = [livro["preco_gbp"] for livro in livros]
contagem = Counter(livro["avaliacao"] for livro in livros if livro["avaliacao"] is not None)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), facecolor="#F4F8FC")
for ax in (ax1, ax2):
    ax.set_facecolor("#F4F8FC")
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.grid(axis="y", color="#DDE6EF", linewidth=0.9)
    ax.set_axisbelow(True)
    ax.tick_params(length=0, pad=8, colors="#476075")

ax1.hist(precos, bins=8, color="#3776AB", edgecolor="#F4F8FC", linewidth=2)
ax1.axvline(LIMITE_PRECO, color="#F28E63", linestyle="--", linewidth=2,
            label=f"Limite: £{LIMITE_PRECO:.0f}")
ax1.set_title("Como estão distribuídos os preços?", loc="left", fontsize=13,
              fontweight="bold", color="#17324D", pad=18)
ax1.set(xlabel="Preço (£)", ylabel="Número de livros")
ax1.legend(frameon=False, loc="upper right")

notas = list(range(1, 6))
barras = ax2.bar([str(nota) for nota in notas], [contagem[nota] for nota in notas],
                 color=["#3776AB", "#3776AB", "#3776AB", "#3776AB", "#20A39E"], width=0.65)
ax2.bar_label(barras, padding=3, color="#17324D")
ax2.set_title("Quantos livros há por avaliação?", loc="left", fontsize=13,
              fontweight="bold", color="#17324D", pad=18)
ax2.set(xlabel="Estrelas", ylabel="Número de livros")

fig.suptitle("LIVROS EM NÚMEROS", fontsize=19, color="#17324D", fontweight="bold", y=1.05)
fig.tight_layout()
plt.show()


### O que aprendemos?

Hoje recolhemos dados de várias páginas, filtrámos livros por preço, criámos um CSV e visualizámos o catálogo em gráficos. O Books to Scrape usa preços e avaliações fictícios para fins de demonstração; estes resultados servem para praticar Python, não para decidir uma compra real.


<h3 style="margin-bottom:5px;">
Tomane Mateus
</h3>

<p style="margin-top:0;">
</p>

<p>
🔗 <b>LinkedIn:</b>
<a href="https://www.linkedin.com/in/tomane-mateus-tomane-7a5205123" target="_blank">
www.linkedin.com/in/tomane-mateus-tomane-7a5205123
</a>
</p>
<p>
💻 <b>GitHub:</b>
<a href="https://github.com/Tomane-Pd" target="_blank">
github.com/Tomane-Pd
</a>
</p>
<p>
🌐 <b>Portfolio:</b>
<a href="https://tomane-portfolio.com" target="_blank">
tomane-portfolio.com
</a>
</p>

<hr style="border:1px solid #ddd; width:100%;">

<p style="color:#555;">
 <b>Siga para mais dicas práticas de Python, IA, Data Science e Automação.</b>
</p>

<p style="font-size:13px;color:#888;margin-top:15px;">
© 2026 <b>Tomane Mateus</b>. Todos os direitos reservados.
</p>

</div>